# NHANES: does disrupted sleep go with lower cognition?

SteadyState rests on one premise: **poor sleep lowers cognitive performance in older adults**.
If that is not true, sleep cannot be used to explain away a drop in speech scores.

This notebook tests it on NHANES 2011–2014 (adults 60+, 7 days of wrist data, 4 cognitive tests).

**Input:** `data/processed/nhanes/nhanes_participants.csv` (from `src/preprocessing/nhanes.py`)

**Questions**
1. Is average sleep duration associated with cognition after adjusting for age, sex, education, race and income?
2. Is **night-to-night variability** in sleep (`sleep_h_sd`) associated with cognition? This is the "state" signal the project relies on.
3. Is sleep **timing** irregularity (`midpoint_h_sd`) associated with cognition?
4. Which cognitive tests drive the association?

All results are **associations, not causes**: cognition was measured once.

In [4]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Find the repo root whether the notebook is run from notebooks/ or the root
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"data"/"processed").exists())
participants = pd.read_csv(ROOT/"data"/"processed"/"nhanes"/"nhanes_participants.csv")
participants.shape

(2853, 36)

## 1. Analysis sample

Keep people with all four cognitive tests and every covariate. Dropping rows with missing covariates
(mostly income) keeps every model on the same people, so their results are comparable.

In [5]:
SLEEP = ["sleep_h_mean", "sleep_h_sd", "midpoint_h_sd"]
COVARIATES = ["age", "female", "education", "race_eth", "income_poverty_ratio", "n_valid_nights"]
TESTS = ["cerad_immediate", "cerad_delayed", "animal_fluency", "digit_symbol"]

d = participants[participants["cog_complete"]==1].dropna(subset=SLEEP+COVARIATES).copy()
print(f"Analysis sample: {len(d):,} of {len(participants):,} participants")

Analysis sample: 2,282 of 2,853 participants


In [6]:
d[["age", "female", "education", "income_poverty_ratio", "self_report_sleep_h",
   "sleep_h_mean", "sleep_h_sd", "midpoint_h_sd", "n_valid_nights", *TESTS]].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,2282.0,69.62,6.76,60.00,64.00,69.00,76.00,80.00
female,2282.0,0.52,0.50,0.00,0.00,1.00,1.00,1.00
education,2282.0,3.41,1.26,1.00,3.00,4.00,4.00,5.00
income_poverty_ratio,2282.0,2.58,1.58,0.00,1.19,2.14,4.08,5.00
self_report_sleep_h,2277.0,7.02,1.44,2.00,6.00,7.00,8.00,12.00
sleep_h_mean,2282.0,8.44,1.63,4.00,7.33,8.26,9.37,14.89
sleep_h_sd,2282.0,1.44,0.64,0.17,0.99,1.34,1.78,5.46
midpoint_h_sd,2282.0,1.08,0.51,0.12,0.73,0.98,1.33,5.78
n_valid_nights,2282.0,6.90,0.92,4.00,6.00,7.00,8.00,8.00
cerad_immediate,2282.0,18.93,4.68,0.00,16.00,19.00,22.00,30.00


## 2. Standardise sleep measures

Each sleep measure is converted to a z-score, so every coefficient below reads the same way:
**change in cognition (in SD units) per 1 SD more of that sleep measure.**

In [7]:
for c in SLEEP:
    d[f"{c}_z"] = (d[c]-d[c].mean())/d[c].std()

ADJ = "age + female + C(education) + C(race_eth) + income_poverty_ratio + n_valid_nights"
SLEEP_TERMS = "sleep_h_mean_z + sleep_h_sd_z + midpoint_h_sd_z"